In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import os, getpass, gc, copy, torch
from torch import nn
from torchmetrics.classification import MulticlassRecall, MulticlassPrecision, MulticlassAccuracy
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device = {str(device).title()}")
script_dir = os.path.dirname(os.path.abspath(os.getcwd))
df_path = os.path.join(script_dir, "Datasets", "Online Content Category.csv")
df = pd.read_csv(df_path)
df = df.dropna()

encoder = LabelEncoder()
x_scaler = StandardScaler()

x = df.drop(columns="target_class").values
y = df["target_class"]
y = encoder.fit_transform(y)
x = x_scaler.fit_transform(x)

x_train,x_test, y_train, y_test = train_test_split(x,y,test_size=0.384, random_state=28)
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

num_features = x_train.shape[1]
num_classes = len(y)

user_account = getpass.getuser()

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

class MultiClassClassifier(nn.Module):
    def __init__(self, input_dim=num_features, num_classes=num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=8),
            nn.LeakyReLU(),
            nn.BatchNorm1d(8),

            nn.Linear(in_features=8, out_features=16),
            nn.LeakyReLU(),
            nn.BatchNorm1d(16),
            
            nn.Linear(in_features=16, out_features=28),
            nn.LeakyReLU(),
            nn.BatchNorm1d(28),

            nn.Linear(in_features=28, out_features=47),
            nn.LeakyReLU(),
            nn.BatchNorm1d(47),
            nn.Dropout(p=0.7),   
            
            nn.Linear(in_features=47, out_features=52),
            nn.ELU(),
            nn.BatchNorm1d(52),
            nn.Dropout(p=0.8),   

            nn.Linear(in_features=52, out_features=68),
            nn.ELU(),
            nn.BatchNorm1d(68),
            nn.Dropout(p=0.3),

            nn.Linear(in_features=68, out_features=42),
            nn.LeakyReLU(),
            nn.BatchNorm1d(42),   

            nn.Linear(in_features=42, out_features=21),
            nn.LeakyReLU(),
            nn.BatchNorm1d(21),

            nn.Linear(in_features=21, out_features=num_classes)
        )     

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return self.network(x)

losses = []
accuracies = []
def gridsearch(epoch_counts_list, rates_list, seed_val):
    best_state = None
    loss_fn = nn.CrossEntropyLoss()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<15} | {'Val Loss':<15} | {'Val Acccuracy':<15}")
    print("="*100) 

    for lr in rates_list:
        for epoch_count in epoch_counts_list:
            torch.manual_seed(int(seed_val))
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(int(seed_val))

            model = MultiClassClassifier().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            for e in tqdm(range(epoch_count)):
                model.train()
                y_preds = model(x_train)
                loss = loss_fn(y_preds, y_train)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            model.eval()
            with torch.inference_mode():
                test_logits = model(x_test)
                test_loss = loss_fn(torch.sigmoid(test_logits), y_test.squeeze().long())
                accuracy = MulticlassAccuracy(num_classes=num_classes).to(device)
                precision = MulticlassPrecision(num_classes=num_classes).to(device)
                recall = MulticlassRecall(num_classes=num_classes).to(device)
                test_acc = accuracy(torch.sigmoid(test_logits), y_test.squeeze().long()).item()
                losses.append(test_loss)
                accuracies.apppend(test_acc)

            print(f"{lr:<15} | {epoch_count:<15} | {test_loss:<15} | {test_acc:<15}")    
            best_state = copy.deepcopy(model.state_dict())
            del model, optimizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print(f"\n● Best Loss = {min(losses):.2f} | Best Accuracy = {max(accuracies):.2f} %")
    print(f"● Worst Loss = {max(losses):.2f} | Worst Accuracy = {min(accuracies):.2f} %")
    print(f"● Precision Score = {precision(torch.sigmoid(test_logits), y_test.squeeze().long()).item():.2f}")
    print(f"● Recall Score = {recall(torch.sigmoid(test_logits), y_test.squeeze().long()).item():.2f}")
    return best_state

epoch_count_list = [980,1230,780,810]
rates = [0.1,0.01,0.001,0.2]
seed_val = float(input("-- Enter the Value for a Seed : ").strip())
best_state = gridsearch(epoch_counts_list=epoch_count_list, rates_list=rates, seed_val=seed_val)
torch.save(best_state, f"Online Content Category Predictor Parameters.pth")

fresh_model = MultiClassClassifier().to(device)
fresh_model.load_state_dict(best_state)
fresh_model.eval()

session_inputs = []
session_predictions = []

conversion = {
    1:"Educational",
    2:"News",
    3:"Entertainment"
}
while True:
    user_name = input("-- Enter your Name ( or type 'exit' to quit ) : ").strip()
    if user_name.lower() == "exit":
        print(f"Exiting Interactive Loop ...")
        break

    word_count = float(input("-- Enter the Word Count  : ").strip())
    image_count = float(input("-- Enter the Images Count : ").strip())
    video_count = float(input("-- Enter the Videos Count : ").strip())
    reading_level_score = float(input("-- Enter the Reading Level Score : ").strip())
    external_links = float(input("-- Enter the Number of External Links : ").strip())
    publish_hour = float(input("-- Enter the Publish Hour : ").strip())

    raw_array = np.array([word_count, image_count, video_count, reading_level_score, external_links, publish_hour])
    scaled_tensor = torch.tensor(x_scaler.transform(raw_array), dtype=torch.float32).to(device)

    with torch.inference_mode():
        logits = fresh_model(scaled_tensor)
        probability = torch.softmax(logits, dim=1)
        pred_idx = torch.argmax(probability, dim=1).item()

    string_prediction = encoder.inverse_transform([pred_idx])[0]
    print(f"Predicted Online Content Category Status for you, {user_name.title()}= {conversion.get(string_prediction, 0)}")    

    session_inputs.append(f"{user_name.title()} ( {word_count} Words )")
    session_predictions.append(conversion.get(string_prediction, 0))

try:
    onedrive = input("-- Do you use One Drive or Not ( Yes / No ) :").strip().title()
    if onedrive == "Yes":
        desktop_path = f"C:/Users/{user_account}/OneDrive/Desktop/Online Content Category Predictor Details.txt"
    elif onedrive == "No":
        desktop_path = f"C:/Users/{user_account}/Desktop/Online Content Category Predictor Details.txt"
except BaseException as e:
    print(f"Invalid Input | Error has Occured = {e}")
try:
    with open(desktop_path, "w", encoding="utf-8") as file:
            file.write(f"== Online Content Category Predictor Details ==\n")   
            file.write(f"\n● Parameter Weights = {fresh_model.network[0].weight[0][0].item():.3f}\n")     
            file.write(f"● Parameter Bias = {fresh_model.network[0].bias[0].item():.3f}\n")     
            file.write(f"● Best Loss = {min(losses):.3f}\n")
            file.write(f"● Worst Loss = {max(losses):.3f}\n")
    print(f"\nSuccessfully Saved the Network Details")
except BaseException as e:
    print(f"Failed to Save Network Details = {e}")        